# IKSP Record Pipeline: run the agents and error scenarios

This notebook runs our institutional AI agents on both error scenarios, with no local setup.

1. Installs Ollama with `nomic-embed-text` (embeddings) and `qwen2.5:7b` (agent reasoning).
2. Clones this repository and the ITU reference code (CrashingGuru/ITUAIReadiness).
3. Loads the **public** knowledge base (6,059 chunks; `knowledge_base/chromadb/`).
4. Runs both scenarios: events released day by day to the right agencies, mandate-scoped retrieval, pipeline tools, policy gate, shared board.
5. Verifies the hash-chained audit logs and shows the transcripts and the interactive replay.

**Runtime:** choose *Runtime → Change runtime type → T4 GPU* if available (about 10 minutes). It also works on CPU, but takes longer.

**Note:** the public knowledge base leaves out copyrighted sources (2,252 chunks), so results can differ slightly from our runs in `results/`, which used the full 8,311-chunk database.

In [ ]:
# 1. Install and start Ollama, then download the two models
import subprocess, time, urllib.request
!apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
subprocess.Popen(['ollama', 'serve'], stdout=open('/content/ollama.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen('http://localhost:11434'); print('Ollama is running'); break
    except Exception:
        time.sleep(2)
!ollama pull nomic-embed-text
!ollama pull qwen2.5:7b

In [ ]:
# 2. Get the code
%cd /content
!rm -rf ITUAIIP ITUAIReadiness
!git clone -q https://github.com/pwdrkg/ITUAIIP.git
!git clone -q https://github.com/CrashingGuru/ITUAIReadiness.git
!pip -q install chromadb ollama pydantic-settings sqlmodel pymupdf pyyaml jsonschema httpx rich networkx

In [ ]:
# 3. Load the public knowledge base into the ITU default folder
!mkdir -p /content/ITUAIReadiness/simulation/data/chromadb
!unzip -q -o /content/ITUAIIP/knowledge_base/chromadb/kb_chromadb_public.zip -d /content/ITUAIReadiness/simulation/data/chromadb
%cd /content/ITUAIIP
!python -c "import sys; sys.path.insert(0,'scripts'); import itu_env; itu=itu_env.load_itu('../ITUAIReadiness'); kb=itu.KnowledgeBase(); print({c: kb.count(c) for c in ['kb_main','kb_usecase']})" 

In [ ]:
# 4. Run both error scenarios (one line per agent step)
!python scripts/run_scenario.py --itu-dir ../ITUAIReadiness --scenario all

In [ ]:
# 5. Verify the audit logs (schema + hash chain)
!python scripts/verify_audit.py

In [ ]:
# 6. Read a transcript (change to policy_reuse for Scenario 2)
from IPython.display import Markdown, display
display(Markdown(open('results/scenarios/mining_renewal/transcript.md', encoding='utf-8').read()))

In [ ]:
# 7. Interactive replay of this run (plus our earlier runs for comparison)
!python docs/build/build_replay.py
import html
from IPython.display import HTML
page = open('docs/replay.html', encoding='utf-8').read()
HTML(f'<iframe srcdoc="{html.escape(page)}" style="width:100%;height:1100px;border:0"></iframe>')

Optional: check the knowledge base itself with `!python scripts/run_retrieval_tests.py --itu-dir ../ITUAIReadiness`. Questions whose expected source is a copyrighted document will miss on the public database; the full-database result is 39/39 (`results/kb_retrieval_results.md`).